# Day 3: from feedback and estimation to a race

Use Day 1's tracking results and Day 2's uncertainty studies to choose a path and speed, then compare the target with a recorded attempt.

| Practical | Minutes | Keep |
| --- | --- | --- |
| R1: course and roles | 15 | Agreed rules, limits and stop operator. |
| R2: shared survey | 40 + 20 retry buffer | Checked gate geometry. |
| R3: two plans | 35 | One speed comparison and margin budget. |
| R4: evidence and dry run | 30 | One result from each earlier day and a checked plan. |
| R5: low-speed check | 30 shared | Configuration-specific go/no-go. |
| R6: race | 100 shared | Configuration, telemetry and observed crossings. |
| R7: analysis | 35 | One plot, explanation and next change. |

**Getting started.** Start `jupyter lab` from the project root and select **Crazyflie lectures (.venv)**; the printed Python path should be inside `.venv`. Run cells with **Shift+Enter**. Paths such as `root / "survey/hoop-1.json"` are relative to the project root.

**Run All** uses a simulated course and a labelled delayed-reference recording. Prefer checked surveys and actual telemetry; after five minutes of acquisition trouble, keep the files and use shared data or the offline example.

Choose a fresh `run_id` for another session (`FileExistsError` protects existing results). After editing an input, rerun from that cell through export. Survey acquisition and flight use the separate tools described below.

In [ ]:
from pathlib import Path
from copy import deepcopy
import subprocess
import sys

# Find the same project and environment used on Days 1 and 2.
root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "code/day3/plan.py").is_file())
sys.path.insert(0, str(root / "code"))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from trajectory_tools.course import construct_course, read_json
from day3.notebook_tools import (
    plan_course, checked_plan, save_plan, clearance_budget, save_example_replay,
)
from trajectory_tools.preview import plot_trajectory
from notebook_radio import check_drone

run_id = "trial-01"  # Change before repeating setup.
output = root / "code/day3/data" / run_id
output.mkdir(parents=True, exist_ok=False)
print("Python:", sys.executable)
print("Results:", output)

## Connect

Put the drone on a stable surface, keep clear of propellers and close cfclient. Set `LABEL` to your drone's sticker label, run the cell and confirm your drone blinked.


In [ ]:
LABEL = None  # Your drone's sticker label: 2 or "A".
uri = None  # Clear the previous selection before reconnecting.
if LABEL is not None:
    uri = check_drone(LABEL, cache_directory=output / ".cflib-cache")


## 1. The course and coordinate frame (R1)

Use the agreed right-handed Lighthouse frame: `[x, y, z]` in metres, Z upward. A gate's unit normal points perpendicular to its opening. Predict the sign for the first crossing.

Select `template_path` below. The default is **simulated back-and-forth motion**; measured courses need measured endpoints and agreed limits.

| Setting | Meaning |
| --- | --- |
| `start`, `finish` | Airborne positions at rest; separate from the resting place for takeoff/landing. |
| `bounds_min`, `bounds_max` | XYZ limits including netting, floor and supports. |
| `target_speed_m_s`, `maximum_speed_m_s` | Requested speed and shared upper limit. |
| `maximum_acceleration_m_s2`, `maximum_tilt_deg` | Acceleration and nominal bank limits. |
| `drone_radius_m`, `clearance_margin_m` | Full cage/deck envelope plus survey, estimation and tracking allowance. |
| `pre_post_distance_m` | Entry/exit distance along the gate normal. |
| `setpoint_rate_hz` | Reference samples per second. |

Edit `template["limits"]` before its display. Retain checked yaw, URI variable, tracking/telemetry thresholds and takeoff/landing durations. Keep shared physical limits fixed.

**Check:** agree rules and roles, including a separate emergency-stop operator. Day 1's position loop produced velocity commands; the race runner sends world XYZ position targets to onboard control. Gains belong to their command interface.

In [ ]:
baseline_plan, candidate_plan = None, None
template, course = None, None
template_path = root / "examples/course.shuttle.synthetic.json"
template = read_json(template_path)
# For a measured course, edit start/finish and the agreed limits here.
# Example: template["limits"]["pre_post_distance_m"] = 0.35
display({key: template[key] for key in ("course_name", "start", "finish", "enabled_for_flight")})
display(template["limits"])


## 2. Measured gates and crossing order (R2)

Share one trace and an independent check per gate across teams. After confirming the blinking drone, run the printed command in a project-root terminal. It includes the discovered URI.

Use `--shape circle --name hoop-1` for a hoop. For a rectangle, use `--shape rectangle --name gate-2` with output `survey/gate-2.json`.

Confirm `STOP AND SURVEY`; wait for **STOP VERIFIED** before handling. The tool verifies `stabilizer.stop=1` and zero motor outputs, leaving the stop engaged.

At `Command [Enter=record, q=quit]:`, press **Enter on the laptop**. Trace the complete inner opening at fixed attitude with Lighthouse receivers visible; press **Enter in that terminal** to stop and confirm `Fit this trace?`. Rectangles need all four edges/corners. Place the drone down before retrying a failure.

Check JSON `centre`, unit `normal` and `radius_m` against manual measurements or an independent pass. Rectangles retain their dimensions/axes and use an inscribed circle for planning. For a fixed-attitude pointer offset, add the world-frame vector from logged point to tip to the fitted centre in a copy; retain the original and correction.

Set `survey_paths` and `crossing_order`: `hoop-1:+` follows the normal and `hoop-1:-` opposes it. Both directions with `laps = 2` give four crossings. Empty lists use the simulated template.

**Check:** another group member checks geometry, direction and margin against the setup. Keep shared checked geometry for retries. Afterwards, place the drone down and **power-cycle** before flight preflight.

In [ ]:
# Run the printed command in a project-root terminal; sampling needs its Enter key.
if uri is not None:
    print("python scripts/measure_hoop.py --uri", uri,
          "--shape circle --name hoop-1 --output survey/hoop-1.json")
    variable = template["uri_environment_variable"]
    print(f"macOS/Linux: export {variable}='{uri}'")
    print(f'PowerShell: $env:{variable} = "{uri}"')
else:
    print("Survey skipped. Select checked shared geometry or keep the simulated template.")


In [ ]:
baseline_plan, candidate_plan = None, None
course = None  # Failed input edits invalidate downstream plans too.
course_provenance = None
survey_paths = []  # Example: [root / "survey/hoop-1.json"]
crossing_order = []  # Example: ["hoop-1:+", "hoop-1:-"]
laps = 2

surveys = [read_json(path) for path in survey_paths]
for survey in surveys:
    display({key: survey[key] for key in ("name", "centre", "normal", "radius_m",
                                          "shape", "width_m", "height_m") if key in survey})
print("Geometry:", "selected survey files; independent checks still needed" if surveys else "TEMPLATE ONLY; no measured survey selected")

course = construct_course(template, surveys, crossing_order, laps)
course_provenance = {
    "source_template": str(template_path),
    "source_surveys": [str(path) for path in survey_paths],
    "crossing_order": list(crossing_order),
    "laps": laps,
}
print("Ordered crossings:", [gate["name"] for gate in course["hoops"]])
print("Enabled for flight:", course["enabled_for_flight"])


**Entry, centre and exit.** With centre C, normal n and distance d, entry is C-d*n and exit is C+d*n. Sketch the first crossing and next turn.

Radial space is `radius_m - drone_radius_m - clearance_margin_m`. The next section checks the template's allowance against earlier evidence.

In [ ]:
gate = course["hoops"][0]
centre = np.asarray(gate["centre"])
normal = np.asarray(gate["normal"])
distance = course["limits"]["pre_post_distance_m"]
entry = centre - distance * normal
exit_point = centre + distance * normal
for label, point in (("entry", entry), ("centre", centre), ("exit", exit_point)):
    print(f"{label:6s} [m]: {point}")
radial_space = (gate["radius_m"] - course["limits"]["drone_radius_m"]
                - course["limits"]["clearance_margin_m"])
print(f"Radial space after drone and margin: {radial_space:.3f} m")


## 3. A continuous trajectory (R3)

Use one result from Day 1's `2Dsim_04_velocity_loop` or `drone_05_tuning`, and one from Day 2 E3-E7, to choose the allowances below. Starting values are illustrative. Day 2's height filter supplies vertical uncertainty; horizontal allowances need separate evidence or stated assumptions.

**Margin.** C is the Day 2 **state-estimation error covariance**; C_position is its position block in m^2. For gate-plane basis U, use U.T @ C_position @ U. The largest eigenvalue gives the widest-axis variance, as in E7:

```text
required margin = survey allowance + tracking allowance + k * plane standard deviation
```

Tracking allowance covers additional control lag/bias; account for overlap with estimation effects. A shadow filter's C needs justification before use for onboard estimates. The sum is a conservative allowance, not an exact collision probability. The planner retains the shared minimum margin and rejects an envelope that cannot fit.

**Motion.** The planner uses quintic segments to join entry, centre and exit with matching position, velocity and acceleration (C2 continuity). Start, finish and shared reversals are at rest. Time stretching by s divides velocity by s and acceleration by s^2 to meet speed, acceleration and tilt bounds. Jerk can jump at joins.

In the preview, grey openings and red normals show crossings; dashed orange envelopes show finite-frame bypass regions. `frame_margin_m` defaults to 0.10 m per side, with drone radius and clearance added. Check supports and raised floors separately. A bypass is not a scored crossing.

**Predict:** which gate consumes most margin, and where does acceleration peak? Printed bounds cover the continuous curve; sampled peaks may be lower. Duration is planned-reference time, excluding takeoff/landing. Check the flight runner's own dry run too.

In [ ]:
baseline_plan, candidate_plan = None, None  # Failed edits invalidate both plans.

# Replace these illustrative allowances with justified values from your earlier work.
position_std_m = np.array([0.01, 0.01, 0.015])
C_position = np.diag(position_std_m**2)
budget = clearance_budget(
    course, C_position, survey_allowance_m=0.01,
    tracking_allowance_m=0.03, sigma_multiplier=2.5,
)
for gate_budget in budget["gates"]:
    print(gate_budget)

# Preserve the source course and bind the chosen budget to the validated plan.
planning_course = deepcopy(course)
planning_course["limits"]["clearance_margin_m"] = max(
    course["limits"]["clearance_margin_m"], budget["required_margin_m"])
planning_provenance = dict(course_provenance, clearance_budget=budget)
baseline_plan = plan_course(planning_course, provenance=planning_provenance)
baseline_course, baseline_trajectory = checked_plan(baseline_plan)
figure = plot_trajectory(baseline_course, baseline_trajectory)
figure.savefig(output / "baseline.png", dpi=140)
display(figure)
plt.close(figure)
print("Chosen margin [m]:", baseline_course["limits"]["clearance_margin_m"])
print("Planned duration [s]:", baseline_trajectory.duration_s)
print("Conservative speed bound [m/s]:", baseline_trajectory.speed_bound_m_s)
print("Conservative acceleration bound [m/s2]:", baseline_trajectory.acceleration_bound_m_s2)

**Change one choice.** Edit `candidate_target_speed` within the shared maximum, keeping geometry and the validated margin budget fixed. Before running, note whether the duration should decrease and which bound might prevent it. Which bound controls it? Time scaling can make different requests produce the same motion.

Approach-distance changes and jerk continuity are optional extensions after this comparison.

In [ ]:
candidate_plan = None  # A failed rerun must not leave an older exportable plan.
candidate, candidate_trajectory = None, None
if baseline_plan is None:
    raise ValueError("Correct the course inputs and rerun baseline planning before comparing a candidate.")
candidate, baseline_trajectory = checked_plan(baseline_plan)  # Independent copy of the validated course.
candidate_target_speed = 0.4  # m/s; choose within the agreed maximum.
candidate["limits"]["target_speed_m_s"] = candidate_target_speed
candidate_plan = plan_course(candidate, provenance=baseline_plan.provenance)
candidate, candidate_trajectory = checked_plan(candidate_plan)

figure = plot_trajectory(candidate, candidate_trajectory)
figure.savefig(output / "candidate.png", dpi=140)
display(figure)
plt.close(figure)
print("Baseline / candidate reference duration [s]:",
      baseline_trajectory.duration_s, candidate_trajectory.duration_s)


**Compare.** `np.linalg.norm(..., axis=1)` gives XYZ speed/acceleration magnitudes. Inspect straights and turns. What changed in timing, and what headroom remains for feedback? For approximately level flight, bank grows as atan(horizontal acceleration / g).

**Check:** explain one difference from your prediction and select a plan using duration, margin and headroom.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for label, plan, style in (("baseline", baseline_plan, "-"),
                           ("candidate", candidate_plan, "--")):
    checked_course, trajectory = checked_plan(plan)
    times, positions, velocities, accelerations = trajectory.sample(
        checked_course["limits"]["setpoint_rate_hz"])
    speed = np.linalg.norm(velocities, axis=1)
    acceleration = np.linalg.norm(accelerations, axis=1)
    ax[0].plot(times, speed, linestyle=style, label=label)
    ax[1].plot(times, acceleration, linestyle=style, label=label)
    print(f"{label}: {trajectory.duration_s:.2f} s, "
          f"sampled peak speed {speed.max():.3f} m/s, "
          f"sampled peak acceleration {acceleration.max():.3f} m/s2")
ax[0].axhline(baseline_course["limits"]["maximum_speed_m_s"], color="k", ls="--")
ax[1].axhline(baseline_course["limits"]["maximum_acceleration_m_s2"], color="k", ls="--")
ax[0].set_ylabel("speed [m/s]")
ax[1].set_ylabel("acceleration [m/s2]")
ax[1].set_xlabel("reference time [s]")
for axis in ax:
    axis.grid()
    axis.legend()
fig.tight_layout()
fig.savefig(output / "comparison.png", dpi=140)
display(fig)
plt.close(fig)


## 4. Save the selected plan and check the evidence (R4)

Use the 30-minute slot for three checks:

1. **10 minutes:** revisit one Day 1 tracking plot and one Day 2 uncertainty/outlier/dropout plot. Explain the margin or speed choice.
2. **10 minutes:** inspect crossing order, supports, bounds and continuous motion limits; select `candidate_plan` or `baseline_plan`.
3. **10 minutes:** compare the flight tool's dry-run preview with this plan. Record any gap and retain the tested baseline if needed.

Use the launch command/directory on the run card. Its radio-free form is `python flight_runner.py --config TEAM_COURSE.json --dry-run --plot`, with an absolute course path. If unavailable, retain the plan and continue offline.

`save_plan` writes disabled `course.json`, `trajectory.csv` (time [s], XYZ position [m], velocity [m/s], acceleration [m/s2]) and `summary.json` (bounds, crossings, input provenance and margin budget). Keep input JSONs. Re-export replaces generated files; failed/edited plans need replanning.

**Check:** the dry run establishes configured geometry and limits. The low-speed flight checks physical tracking and visibility.

In [ ]:
selected_plan = candidate_plan  # Or baseline_plan.
summary = save_plan(selected_plan, output)
print("Saved disabled plan:", output / "course.json")
for crossing in summary["crossings"]:
    print(f"{crossing['name']}: {crossing['time_s']:.2f} s")


## 5. The flight (R5-R6)

Use the run card's runner/configuration and export the discovered URI as printed above. Agree readiness within the group; assign pilot, separate emergency-stop operator, telemetry monitor and recorder.

Before the low-speed test, cross-check supports/floor changes, full drone envelope, Lighthouse visibility, battery, measured mass/controller settings, URI, command interface, limits, tracking and landing. After handling, place the drone down, power-cycle to clear the firmware stop and repeat preflight.

Allow one short low-speed check per ready team, with handovers/recovery. Use the checked baseline or shared data if the candidate/equipment is not ready.

Freeze estimator settings at **12:30**, controller settings at **13:00**, and all algorithm changes at **13:30**. Race **14:00-15:40**, then protect the 35-minute analysis slot. Two attempts are useful when capacity allows.

**Check:** confirm the pilot/stop operator, and save each attempt's exact configuration, telemetry and observed crossings, including aborts. Everyone can call a stop. Onboard estimation/control drives the motors; Day 2's laptop filter is diagnostic in the baseline.

## 6. Recorded motion and tracking difference (R7)

Select telemetry and its **exact flown course**, or shared checked data. With both paths empty, the cell creates a labelled example: selected-plan positions delayed by 0.12 s. It prescribes positions for plotting practice rather than predicting drone performance.

CSV `event=position` rows carry `x_m,y_m,z_m`; `event=command` rows carry `command_x_m,command_y_m,command_z_m` and `phase`. The plotter uses `host_monotonic_s`, selects `race`, and matches estimates to the last command, up to 0.25 s old.

The estimated-minus-commanded difference is e(t)=p_onboard(t)-p_command(t). Day 2's predicted residual compares a measurement with its filter prediction. Estimation and timestamp alignment also contribute to e.

Spend **5 minutes selecting data, 20 minutes inspecting the path/errors, and 10 minutes discussing one change**. Predict lag's sign on outbound/return legs; compare turns, straights and clearance. Use observed crossings and timing to judge a race result; identify a single attempt as such.

**Check:** name the data source, explain one feature and keep one plot with your next change.

In [ ]:
telemetry_csv = None  # Prefer an actual attempt: root / "data/race-telemetry.csv".
flown_course_json = None  # The exact course used for that attempt.
recording_source = None

if (telemetry_csv is None) != (flown_course_json is None):
    raise ValueError("Select both the telemetry and the exact flown course, or neither.")

if telemetry_csv is None:
    telemetry_path, course_path = save_example_replay(selected_plan, output / "example-replay")
    recording_source = "SYNTHETIC delayed-reference example (0.12 s lag)"
    prefix = output / "synthetic"
else:
    telemetry_path, course_path = Path(telemetry_csv), Path(flown_course_json)
    recording_source = "Selected recording; retain its source and matching course"
    prefix = output / "flown"

print("Data source:", recording_source)
subprocess.run([sys.executable, str(root / "scripts/plot_race.py"),
                str(telemetry_path), "--course", str(course_path),
                "--phase", "race", "--output", str(prefix), "--no-show"], check=True)
display(Image(filename=f"{prefix}-trajectory.png"))
display(Image(filename=f"{prefix}-errors.png"))

## Discuss and keep

Keep the notebook, surveys, selected plan and one useful plot. Add the margin/speed choice, data source, outcome and next change to the team's existing notes.

For optional further work before the relevant freeze, see the [stretch exercises](../../docs/race_practical.md#stretch-exercises).

## If a cell fails

| Symptom | Next step |
| --- | --- |
| Import/root error | Check kernel and project-root launch. |
| Existing output | Choose a fresh `run_id`. |
| Invalid budget/plan | Check units, allowances, geometry and limits; rerun from the edited input through export. |
| Capture or flight check fails | Keep the failed data; use checked shared data, the labelled example or a checked baseline. |

## Reference and optional material

The notebook contains the core steps. Look up these details when needed:
- [Flight checks and deadlines](../../docs/race_practical.md).
- [Survey procedure](../../docs/hoop_survey_challenge.md).
- [Course fields and flight-tool handover](../../docs/course_interface.md).
- [Recording/plotting formats](../../docs/post_flight_analysis.md).

**Optional background and extensions:**
- [Planner mathematics, limitations and jerk-continuity work](../../docs/trajectory_construction.md).
- [Control and estimation equations](../../docs/quadrotor_equations_of_motion.md).